# 16 · GSE232381 · bulk_RNA_seq · module–trait associations

Reads `expression.rds`, `modules.rds`, `metadata.rds`, and the GSE65391 end product `module_genes.csv`.
Writes the end products `module_traits.csv` and `clinical_dictionary.csv` in `data/run_artifacts/GSE232381/`.

**Trait:** `nephritis_active`, 1 for active and 0 for inactive lupus nephritis (10 and 6 samples).

**Eigengenes**, two families:
1. **array modules**: each GSE65391 module's genes, as measured here, summarised by their first
   principal component (`WGCNA::moduleEigengenes`); these are the ones compared across studies;
2. **own modules**: the exploratory modules of notebook 14.

**Test.** Pearson (point-biserial) r between eigengene and `nephritis_active`, p from `cor.test`, BH q
within each family. n = 16. **A finding** is q < 0.05.

In [1]:
if (!requireNamespace("WGCNA", quietly = TRUE)) install.packages("WGCNA", repos = "https://cloud.r-project.org")
suppressMessages(library(WGCNA))
source("../src/paths.R")
x    <- readRDS(art("GSE232381", "expression.rds"))
own  <- readRDS(art("GSE232381", "modules.rds"))
meta <- readRDS(art("GSE232381", "metadata.rds"))
mg   <- read.csv(art("GSE65391", "module_genes.csv"))
datExpr <- t(x$E)
mg <- mg[mg$gene %in% colnames(datExpr), ]
ME_array <- moduleEigengenes(datExpr[, mg$gene], colors = mg$module)$eigengenes
ME_own   <- own$MEs[, colnames(own$MEs) != "MEgrey"]
trait <- meta[rownames(datExpr), "ln_active"]
table(nephritis_active = trait)

nephritis_active
 0  1 
 6 10 

In [2]:
test_family <- function(ME, family) {
  out <- do.call(rbind, lapply(colnames(ME), function(me) {
    ct <- cor.test(ME[, me], trait)
    data.frame(family = family, module = sub("^ME", "", me), trait = "nephritis_active", n = length(trait),
               r = unname(ct$estimate), p = ct$p.value)
  }))
  out$q <- p.adjust(out$p, "BH"); out[order(out$p), ]
}
res_array <- test_family(ME_array, "array module")
res_own   <- test_family(ME_own, "own module")
format(res_array, digits = 3)
head(format(res_own, digits = 3), 10)
c(array_modules_q_below_0.05 = sum(res_array$q < 0.05), own_modules_q_below_0.05 = sum(res_own$q < 0.05))

,family,module,trait,n,r,p,q
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
1,array module,black,nephritis_active,16,-0.4657,0.0691,0.570
13,array module,salmon,nephritis_active,16,-0.3744,0.1531,0.570
15,array module,turquoise,nephritis_active,16,0.3306,0.2110,0.570
3,array module,brown,nephritis_active,16,-0.3282,0.2146,0.570
7,array module,lightcyan,nephritis_active,16,-0.3219,0.2240,0.570
6,array module,greenyellow,nephritis_active,16,-0.3061,0.2489,0.570
12,array module,red,nephritis_active,16,-0.3057,0.2496,0.570
8,array module,magenta,nephritis_active,16,0.2647,0.3218,0.572
11,array module,purple,nephritis_active,16,-0.2497,0.3511,0.572


,family,module,trait,n,r,p,q
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
16,own module,yellow,nephritis_active,16,-0.66827,0.00466,0.0979
31,own module,violet,nephritis_active,16,0.64504,0.00697,0.0979
32,own module,greenyellow,nephritis_active,16,0.61625,0.01102,0.0979
15,own module,red,nephritis_active,16,-0.61329,0.01152,0.0979
30,own module,lightyellow,nephritis_active,16,0.53787,0.03163,0.2151
34,own module,magenta,nephritis_active,16,0.51448,0.04145,0.2349
17,own module,darkgrey,nephritis_active,16,0.44749,0.08220,0.3866
26,own module,darkred,nephritis_active,16,0.42393,0.10175,0.3866
12,own module,lightcyan,nephritis_active,16,-0.42327,0.10235,0.3866


array_modules_q_below_0.05   own_modules_q_below_0.05 
                         0                          0

**Result.** No module, array or own, is associated with nephritis activity at q < 0.05. The largest
array-module effect is black, r = −0.47 (p = 0.07). With 16 samples, only |r| above about 0.5 can
reach p < 0.05 before correction.

In [3]:
res <- rbind(res_array, res_own)
res$fisher_z <- atanh(res$r); res$se <- 1 / sqrt(res$n - 3)
write.csv(res, art("GSE232381", "module_traits.csv"), row.names = FALSE)
write.csv(data.frame(study = "GSE232381", trait = "nephritis_active", recorded = 16, binary = TRUE),
          art("GSE232381", "clinical_dictionary.csv"), row.names = FALSE)